# Classical ML Baselines

RF, XGBoost, LightGBM (Decision #14) plus the naive/trivial baseline predictor (Decision #17), under the fixed tuning budget (Decision #5).

**Scope for this pass:** F-DATA's `duration` (execution time) target only. EXPERIMENT_TRACKER.md's notebook 05 row lists only RF/XGBoost/LightGBM without naming a dataset (unlike notebook 03's row, which explicitly named both F-DATA and PM100) — treated here as PM100 and F-DATA's other two targets (memory, power) being deferred to a follow-up pass rather than assumed in scope, see Summary below.

See the conceptualization plan and EXPERIMENT_TRACKER.md for full context.

In [1]:
import sys
sys.path.append("..")

from src import baselines, config, features, metrics, models, plotting, roofline, splits

## Config

In [2]:
import glob
import time

import numpy as np
import optuna
import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

optuna.logging.set_verbosity(optuna.logging.WARNING)

FDATA_DIR = "../data/raw/fdata"
N_CONSECUTIVE_MONTHS = 6  # same dev-scale pattern as notebook 03 (contiguous, real time range)
SAMPLE_SIZE = 1_000_000   # train rows (Decision #10) -- see EXPERIMENT_TRACKER.md sample-size reminder
TARGET_COL = "duration"   # F-DATA execution time (Decision #2)
N_TRIALS = models.TuningBudget().n_trials  # Decision #5 fixed tuning budget, same for every model family
SEED = 0
N_BUCKETS = 5              # job-size stratification buckets (cnumr-based)
EMBEDDING_VARIANCE_THRESHOLD = 0.90
TUNE_TRAIN_FRAC = 0.8      # chronological carve-out WITHIN train_sample: first 80% tunes, last 20% validates

print(f"SAMPLE_SIZE={SAMPLE_SIZE:,}  N_TRIALS={N_TRIALS}  TARGET_COL={TARGET_COL}")

SAMPLE_SIZE=1,000,000  N_TRIALS=50  TARGET_COL=duration


/home/tobi/Documents/thesis/code/hpc-dl-enhanced/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 1: Load 6 consecutive F-DATA months + exclude non-completed jobs (Decision #4)

Same 6-month contiguous slice notebook 03 used for its Roofline construction — a real, contiguous time range is needed for the chronological split regardless of the eventual sample size.

In [3]:
fdata_files = sorted(glob.glob(f"{FDATA_DIR}/*.parquet"))[:N_CONSECUTIVE_MONTHS]
print(f"Loading {len(fdata_files)} months: {[f.split('/')[-1] for f in fdata_files]}")

fdata = features.load_fdata_no_embedding(fdata_files)
fdata = features.filter_completed_jobs(fdata, "fdata")
print(f"rows after filtering: {len(fdata):,}")

Loading 6 months: ['21_03.parquet', '21_04.parquet', '21_05.parquet', '21_06.parquet', '21_07.parquet', '21_08.parquet']


[fdata] excluded 10.5% of jobs as non-completed (304610 / 2897734)
rows after filtering: 2,593,124


## Step 2: `mszl` sentinel fix (Decision #19)

`mszl` (memory size limit requested, Tier A) uses an unsigned-int sentinel (`2**64 - 1`) for "no limit requested" rather than a null — found while scratch-timing this notebook's `SAMPLE_SIZE` decision, where it broke XGBoost/LightGBM's histogram binning (see EXPERIMENT_TRACKER.md Data Gotchas). Fixed for real here via `src/features.py`, not the scratch script's throwaway exclusion.

In [4]:
n_sentinel_before = int((fdata["mszl"] >= 1e15).sum())
print(f"mszl sentinel rows before fix: {n_sentinel_before:,} / {len(fdata):,} "
      f"({n_sentinel_before / len(fdata):.1%})")

fdata = features.handle_mszl_sentinel(fdata)
features.assert_mszl_sanitized(fdata)

print(f"mszl range after fix: [{fdata['mszl'].min():.3g}, {fdata['mszl'].max():.3g}]")
print(f"mszl_unlimited True: {fdata['mszl_unlimited'].sum():,} ({fdata['mszl_unlimited'].mean():.1%})")

mszl sentinel rows before fix: 2,578,047 / 2,593,124 (99.4%)


mszl range after fix: [0, 3.06e+10]
mszl_unlimited True: 2,578,047 (99.4%)


## Step 3: Historical rolling-stat feature (Tier A, Decision #1)

Computed on the full chronologically-ordered dataset, **before** the split/sample below. The function itself only ever looks backward (`shift(1)` before the rolling window), so this doesn't leak future information into training either way — the reason to do it first is so a job on the *test* side of the later chronological split still gets credit for that user's real training-period history, instead of an artificially truncated one computed only from other test-period rows.

In [5]:
fdata = features.add_user_rolling_stat(fdata, "fdata", TARGET_COL, window=5)
rolling_col = f"{TARGET_COL}_user_rolling_mean"
print(f"rolling stat available for {fdata[rolling_col].notna().sum():,} / {len(fdata):,} jobs")

rolling stat available for 2,592,189 / 2,593,124 jobs


## Step 4: Chronological split — before any sampling

`src/splits.py`'s `chronological_split`, the same train/test boundary utility every notebook from 03 onward reuses. Splitting happens **before** any row sampling: stratify-sampling first would risk letting the draw disturb which rows fall before/after the time boundary the whole evaluation depends on.

In [6]:
train_df, test_df = splits.chronological_split(fdata, "fdata")
split_ratio = len(test_df) / len(train_df)
print(f"train={len(train_df):,} test={len(test_df):,} (test/train ratio={split_ratio:.4f})")

train=1,815,596 test=777,528 (test/train ratio=0.4282)


## Step 5: Stratified sampling by job-size bucket — independently within train and within test

`test_sample` is sized proportionally to the split ratio just measured above (not also fixed at `SAMPLE_SIZE`).

In [7]:
test_target_n = round(SAMPLE_SIZE * split_ratio)

train_sample = features.stratified_sample_by_job_size(train_df, "fdata", SAMPLE_SIZE, n_buckets=N_BUCKETS, seed=SEED)
test_sample = features.stratified_sample_by_job_size(test_df, "fdata", test_target_n, n_buckets=N_BUCKETS, seed=SEED)

print(f"train_sample={len(train_sample):,} (target {SAMPLE_SIZE:,}), "
      f"test_sample={len(test_sample):,} (target {test_target_n:,})")

train_sample=1,000,000 (target 1,000,000), test_sample=428,249 (target 428,249)


## Step 6: Tier A feature matrix + leakage re-check (Decision #19)

Re-running the leakage sanity check on the post-sampling data — this should still pass (sampling doesn't touch tier membership), but re-verifying after every transformation step is the point of Decision #19.

In [8]:
train_tier_a = features.build_tier_a_features(train_sample, "fdata", include_embedding=False)
test_tier_a = features.build_tier_a_features(test_sample, "fdata", include_embedding=False)

features.assert_no_tier_leakage(list(train_tier_a.columns), "A", "fdata")
features.assert_no_tier_leakage(list(test_tier_a.columns), "A", "fdata")
print(f"F-DATA Tier A: {len(train_tier_a.columns)} columns, OK")

F-DATA Tier A: 13 columns, OK


## Step 7: Embedding PCA — fit on TRAIN only, transform both

`jid` is **not** a unique key across this 6-month load (verified during the scratch timing check: 1,649,605 unique / 2,897,734 rows) — cannot be used to re-select rows for a separate embedding read. Loading the embedding column once, in the same file order + `ignore_index=True` as `load_fdata_no_embedding`, keeps its positional index aligned with `fdata`/`train_df`/`test_df` directly, so `.loc[sample.index]` is safe. Component count is picked empirically (90% explained variance on `train_sample` itself), not copied from notebook 02's dev-sample number — a different sample needs its own check.

In [9]:
embed_full = pd.concat(
    [pd.read_parquet(f, columns=["embedding"]) for f in fdata_files], ignore_index=True
)
embed_train = embed_full.loc[train_sample.index]
embed_test = embed_full.loc[test_sample.index]

pca_check = features.compute_embedding_explained_variance(embed_train)
n_components = features.n_components_for_variance(pca_check, threshold=EMBEDDING_VARIANCE_THRESHOLD)
print(f"embedding PCA: {n_components} components reach {EMBEDDING_VARIANCE_THRESHOLD:.0%} "
      f"explained variance (fit on train_sample only, {len(embed_train):,} rows)")

embedding_pca = features.fit_fdata_embedding_pca(embed_train, n_components=n_components)
embed_train_pca = features.transform_fdata_embedding(embed_train, embedding_pca)
embed_test_pca = features.transform_fdata_embedding(embed_test, embedding_pca)

embedding PCA: 37 components reach 90% explained variance (fit on train_sample only, 1,000,000 rows)


## Step 8: Final numeric feature matrix + target transform (Decision #3)

`build_fdata_numeric_matrix` (new in `src/features.py`) turns the raw Tier A frame into something RF/XGBoost/LightGBM can fit on: numeric passthrough, `mszl_unlimited` as int, datetimes to epoch seconds via `datetime_to_epoch_seconds` (not a blind divisor — see its docstring for the microsecond-vs-nanosecond trap this avoids), `jobenv_req` factorized, `usr` frequency-encoded, `jnam` dropped (captured via the PCA embedding instead), plus the rolling-stat column from Step 3. Remaining NaNs (mostly the rolling-stat column for users with no prior training-period history) are filled with 0 — RF has no native NaN handling, so all three model families need a finite matrix; the missing-history signal itself isn't separately flagged here since it's expected to be common at this dev-scale slice (see EXPERIMENT_TRACKER.md's rolling-stat coverage note).

In [10]:
X_train = features.build_fdata_numeric_matrix(
    train_tier_a, embed_train_pca, extra_columns=train_sample[[rolling_col]]
).fillna(0.0)
X_test = features.build_fdata_numeric_matrix(
    test_tier_a, embed_test_pca, extra_columns=test_sample[[rolling_col]]
).fillna(0.0)

y_train_raw = train_sample[TARGET_COL].to_numpy(dtype=float)
y_test_raw = test_sample[TARGET_COL].to_numpy(dtype=float)
y_train = features.transform_target(y_train_raw)
y_test = features.transform_target(y_test_raw)

metrics.expm1_round_trip_check(y_train_raw)
metrics.expm1_round_trip_check(y_test_raw)

print(f"X_train shape: {X_train.shape}  X_test shape: {X_test.shape}")
print(f"feature columns: {list(X_train.columns)}")

X_train shape: (1000000, 50)  X_test shape: (428249, 50)
feature columns: ['cnumr', 'nnumr', 'elpl', 'mszl', 'pri', 'freq_req', 'mszl_unlimited', 'adt_epoch', 'qdt_epoch', 'schedsdt_epoch', 'jobenv_req_code', 'usr_freq', 'emb_pc_0', 'emb_pc_1', 'emb_pc_2', 'emb_pc_3', 'emb_pc_4', 'emb_pc_5', 'emb_pc_6', 'emb_pc_7', 'emb_pc_8', 'emb_pc_9', 'emb_pc_10', 'emb_pc_11', 'emb_pc_12', 'emb_pc_13', 'emb_pc_14', 'emb_pc_15', 'emb_pc_16', 'emb_pc_17', 'emb_pc_18', 'emb_pc_19', 'emb_pc_20', 'emb_pc_21', 'emb_pc_22', 'emb_pc_23', 'emb_pc_24', 'emb_pc_25', 'emb_pc_26', 'emb_pc_27', 'emb_pc_28', 'emb_pc_29', 'emb_pc_30', 'emb_pc_31', 'emb_pc_32', 'emb_pc_33', 'emb_pc_34', 'emb_pc_35', 'emb_pc_36', 'duration_user_rolling_mean']


## Step 9: Naive per-user-median baseline (Decision #17)

In [11]:
user_medians, global_median = baselines.fit_naive_baseline(train_sample, "fdata", TARGET_COL)
naive_pred = baselines.predict_naive_baseline(test_sample, "fdata", user_medians, global_median)
naive_metrics = metrics.regression_metrics(y_test_raw, naive_pred)

print("Naive per-user-median baseline (this notebook's sampled test set):")
for k, v in naive_metrics.items():
    print(f"  {k}: {v:,.4f}")

Naive per-user-median baseline (this notebook's sampled test set):
  MAE: 9,860.9922
  RMSE: 19,988.5505
  R2: -0.0907
  MAPE: 15,997.9017


## Step 10: Chronological tuning/validation carve-out within train (never touches test)

Optuna needs a validation objective distinct from the final test evaluation — tuning hyperparameters directly against `test_sample`'s score would invalidate the headline numbers below. Reusing the same `chronological_split` utility on `train_sample` alone (never on `test_sample`) gives a proper time-ordered validation split for the search, keeping `test_sample` untouched until Step 12's final fit.

In [12]:
tune_train_raw, tune_val_raw = splits.chronological_split(train_sample, "fdata", train_frac=TUNE_TRAIN_FRAC)
print(f"tune_train={len(tune_train_raw):,} tune_val={len(tune_val_raw):,}")

tune_train_tier_a = features.build_tier_a_features(tune_train_raw, "fdata", include_embedding=False)
tune_val_tier_a = features.build_tier_a_features(tune_val_raw, "fdata", include_embedding=False)

embed_tune_train = embed_train.loc[tune_train_raw.index]
embed_tune_val = embed_train.loc[tune_val_raw.index]
embed_tune_train_pca = features.transform_fdata_embedding(embed_tune_train, embedding_pca)
embed_tune_val_pca = features.transform_fdata_embedding(embed_tune_val, embedding_pca)

X_tune_train = features.build_fdata_numeric_matrix(
    tune_train_tier_a, embed_tune_train_pca, extra_columns=tune_train_raw[[rolling_col]]
).fillna(0.0)
X_tune_val = features.build_fdata_numeric_matrix(
    tune_val_tier_a, embed_tune_val_pca, extra_columns=tune_val_raw[[rolling_col]]
).fillna(0.0)

y_tune_train = features.transform_target(tune_train_raw[TARGET_COL].to_numpy(dtype=float))
y_tune_val = features.transform_target(tune_val_raw[TARGET_COL].to_numpy(dtype=float))

print(f"X_tune_train shape: {X_tune_train.shape}  X_tune_val shape: {X_tune_val.shape}")

tune_train=800,000 tune_val=200,000


X_tune_train shape: (800000, 50)  X_tune_val shape: (200000, 50)


## Step 11: Optuna hyperparameter search — RF / XGBoost / LightGBM (Decision #5)

Fixed budget: `N_TRIALS` Optuna trials for every model family, same fixed seed for the sampler — the fairness principle Decision #5 requires. Each family's own search-space *ranges* differ (they have different hyperparameters entirely), which is an ordinary per-algorithm modeling choice, not a fairness parameter — only the trial *count* is held fixed across families.

In [13]:
def rf_objective(trial):
    params = dict(
        n_estimators=trial.suggest_int("n_estimators", 50, 250),
        max_depth=trial.suggest_int("max_depth", 4, 18),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 20),
        max_features=trial.suggest_float("max_features", 0.3, 1.0),
    )
    model = RandomForestRegressor(**params, n_jobs=-1, random_state=SEED)
    model.fit(X_tune_train, y_tune_train)
    pred = model.predict(X_tune_val)
    return float(np.sqrt(np.mean((y_tune_val - pred) ** 2)))


def xgb_objective(trial):
    params = dict(
        n_estimators=trial.suggest_int("n_estimators", 50, 500),
        max_depth=trial.suggest_int("max_depth", 3, 12),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        subsample=trial.suggest_float("subsample", 0.5, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
    )
    model = XGBRegressor(**params, tree_method="hist", n_jobs=-1, random_state=SEED)
    model.fit(X_tune_train, y_tune_train)
    pred = model.predict(X_tune_val)
    return float(np.sqrt(np.mean((y_tune_val - pred) ** 2)))


def lgbm_objective(trial):
    params = dict(
        n_estimators=trial.suggest_int("n_estimators", 50, 500),
        num_leaves=trial.suggest_int("num_leaves", 15, 255),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        subsample=trial.suggest_float("subsample", 0.5, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
    )
    # bagging_freq must be >0 for `subsample` to have any effect in LightGBM
    # (it silently no-ops otherwise) -- fixed, not tuned, so the tuned
    # subsample value actually does something.
    model = LGBMRegressor(**params, bagging_freq=1, n_jobs=-1, random_state=SEED, verbose=-1)
    model.fit(X_tune_train, y_tune_train)
    pred = model.predict(X_tune_val)
    return float(np.sqrt(np.mean((y_tune_val - pred) ** 2)))


objectives = {"RandomForest": rf_objective, "XGBoost": xgb_objective, "LightGBM": lgbm_objective}
best_params = {}
tuning_seconds = {}

for name, objective in objectives.items():
    start = time.perf_counter()
    study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)
    tuning_seconds[name] = time.perf_counter() - start
    best_params[name] = study.best_params
    print(f"{name}: best RMSE(log-space)={study.best_value:.4f}  "
          f"tuning time={tuning_seconds[name]:.1f}s  n_trials={N_TRIALS}")
    print(f"    best_params={study.best_params}")

RandomForest: best RMSE(log-space)=1.3253  tuning time=2870.3s  n_trials=50
    best_params={'n_estimators': 68, 'max_depth': 5, 'min_samples_leaf': 9, 'max_features': 0.3062286801416294}


XGBoost: best RMSE(log-space)=1.3974  tuning time=294.5s  n_trials=50
    best_params={'n_estimators': 376, 'max_depth': 4, 'learning_rate': 0.015356882409770645, 'subsample': 0.5929438626292725, 'colsample_bytree': 0.7571405111292817}


LightGBM: best RMSE(log-space)=1.3714  tuning time=259.2s  n_trials=50
    best_params={'n_estimators': 183, 'num_leaves': 49, 'learning_rate': 0.011213439035811911, 'subsample': 0.6251683339679805, 'colsample_bytree': 0.6672496721697027}


## Step 12: Final fit on the full `train_sample` with the best hyperparameters, evaluate on `test_sample`

One fit per model at the winning configuration — not a repeated search loop, and not yet the multi-seed repeat Decision #6 calls for (that's notebook 08's job; this is a single-seed headline result).

In [14]:
final_models = {
    "RandomForest": RandomForestRegressor(**best_params["RandomForest"], n_jobs=-1, random_state=SEED),
    "XGBoost": XGBRegressor(**best_params["XGBoost"], tree_method="hist", n_jobs=-1, random_state=SEED),
    "LightGBM": LGBMRegressor(**best_params["LightGBM"], bagging_freq=1, n_jobs=-1, random_state=SEED, verbose=-1),
}

fit_seconds = {}
results = {}
for name, model in final_models.items():
    start = time.perf_counter()
    model.fit(X_train, y_train)
    fit_seconds[name] = time.perf_counter() - start
    pred_log = model.predict(X_test)
    pred_raw = features.inverse_transform_target(pred_log)
    results[name] = {
        "log_space": metrics.regression_metrics(y_test, pred_log),
        "raw_space": metrics.regression_metrics(y_test_raw, pred_raw),
    }
    print(f"{name}: final fit time={fit_seconds[name]:.1f}s")

RandomForest: final fit time=19.2s


XGBoost: final fit time=6.3s


LightGBM: final fit time=5.3s


## Step 13: Results — F-DATA `duration` target

In [15]:
print("=" * 70)
print(f"RESULTS -- F-DATA {TARGET_COL} target, test_sample (n={len(test_sample):,})")
print("=" * 70)

summary_rows = [{"model": "Naive (per-user median)", **naive_metrics}]
for name in final_models:
    summary_rows.append({"model": name, **results[name]["raw_space"]})
summary_df = pd.DataFrame(summary_rows).set_index("model")

print(f"\nRaw-space (seconds) metrics, this notebook's SAMPLE_SIZE={SAMPLE_SIZE:,} sample:")
print(summary_df.round(4).to_string())

log_rows = [{"model": name, **results[name]["log_space"]} for name in final_models]
log_df = pd.DataFrame(log_rows).set_index("model")
print("\nLog-space metrics (Decision #3 back-transform check):")
print(log_df.round(4).to_string())

print("\n" + "-" * 70)
print("Reference only -- notebook 03's analytical baselines, FULL 6-month "
      "chronological split (no SAMPLE_SIZE subsampling; different sample "
      "scope, not directly comparable row-for-row, shown for context):")
print("-" * 70)
reference_df = pd.DataFrame([
    {"model": "Roofline (notebook 03, full split)", "MAE": 10921.0303, "RMSE": 22019.9207, "R2": -0.3262, "MAPE": 99.9303},
    {"model": "Naive (notebook 03, full split)", "MAE": 9868.2419, "RMSE": 20014.9060, "R2": -0.0957, "MAPE": 15770.4878},
]).set_index("model")
print(reference_df.to_string())

print("\nOptuna tuning time per model family (N_TRIALS={} each):".format(N_TRIALS))
for name in final_models:
    print(f"  {name}: tuning={tuning_seconds[name]:.1f}s  final_fit={fit_seconds[name]:.1f}s")

RESULTS -- F-DATA duration target, test_sample (n=428,249)

Raw-space (seconds) metrics, this notebook's SAMPLE_SIZE=1,000,000 sample:
                               MAE        RMSE      R2        MAPE
model                                                             
Naive (per-user median)  9860.9922  19988.5505 -0.0907  15997.9017
RandomForest             7883.6050  18230.3750  0.0927    609.1837
XGBoost                  5801.8282  13233.0082  0.5220    749.8199
LightGBM                 7319.7879  16630.1700  0.2450    490.8782

Log-space metrics (Decision #3 back-transform check):
                 MAE    RMSE      R2     MAPE
model                                        
RandomForest  0.8207  1.1574  0.8084  18.0796
XGBoost       0.6136  0.9014  0.8838  11.7101
LightGBM      0.9273  1.1165  0.8217  19.3689

----------------------------------------------------------------------
Reference only -- notebook 03's analytical baselines, FULL 6-month chronological split (no SAMPLE_SIZE sub

## Summary

Pipeline executed: load 6 consecutive F-DATA months -> exclude non-completed jobs (Decision #4) -> fix the `mszl` sentinel for real (Decision #19, `src/features.py`) -> historical rolling-stat feature computed on the full chronological timeline (Decision #1) -> chronological split (`src/splits.py`) -> stratified sampling by job-size bucket, independently within train and within test, at `SAMPLE_SIZE` and the proportional test size (Decision #10) -> Tier A feature matrix + leakage re-check (Decision #19) -> embedding PCA fit on train only, component count chosen empirically (Decision #7) -> full numeric encoding (`build_fdata_numeric_matrix`, new) -> target log1p transform + round-trip check (Decision #3) -> naive per-user-median baseline (Decision #17) -> chronological tune/validation carve-out within train, never touching test -> Optuna hyperparameter search, `N_TRIALS` trials per model family (Decision #5) -> single final fit per model at the winning configuration, evaluated on `test_sample` (Decision #14).

**Deferred, not in scope for this pass:**
- **PM100.** EXPERIMENT_TRACKER.md's notebook 05 row lists only RF/XGBoost/LightGBM with no dataset named, unlike notebook 03's row which explicitly covered both F-DATA and PM100 — treated as an open item to confirm rather than assumed in scope here, since PM100 would need its own Tier A numeric encoding (different columns, no `mszl`/embedding) and its own naive baseline/Optuna sweep.
- **F-DATA's `memory` (`mmszu`) and `power` (`avgpcon`) targets.** This pass covers `duration` only; the same pipeline structure applies to the other two targets but each needs its own Optuna sweep (three separate searches per model family, per Decision #5's "3 targets" scope).
- **Decision #6's multi-seed repeats.** The results above are a single seed's fit, not the mean +/- std across `TuningBudget.seeds` that Decision #6 and notebook 08's statistical rigor pass call for.
- **Full-scale F-DATA (all 38 months, no subsampling).** Decision #10's fallback path, only pursued if time/compute allow later in the timeline.